### Notebook 3: Building a Neural Network
We will define a network to process the MNIST dataset and look into every step of its **forwars** pass

As a reminder: the shape of our images was 28x28


In [28]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [29]:
# Set which device we will use
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

Using device: cpu


### Definition of the class and initialization
We are free to play with the hidden dimensions of our network, but we need to pay attention to:
* **input size**, it should match the size of our input images
* **output size** it should match the size of our label tensor

In [38]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()             # 2D image flattened to 1D tensor
        self.linear_relu_stack = nn.Sequential( # Sequential container
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [39]:
# Create an instance of the NeuralNetwork class
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
)


In [53]:
# Print model parameters
print("Model structure: ", model, "\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure:  NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=10, bias=True)
  )
) 


Layer: linear_relu_stack.0.weight | Size: torch.Size([128, 784]) | Values : tensor([[ 2.4514e-02, -2.7976e-02,  6.9695e-03,  ..., -5.8510e-05,
          2.4460e-02,  2.1762e-02],
        [-7.3676e-03,  2.7214e-02, -1.4045e-03,  ..., -2.6185e-02,
          2.2548e-03, -1.2886e-02]], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([128]) | Values : tensor([0.0337, 0.0254], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([128, 128]) | Values : tensor([[ 5.1440e-02, -7.5831e-02,  2.3202e-02,  7.9188e-02, -1.3986e-03,
          4.2596e-02, -7.1042e-02,  4.7555e-02, -6.3095e-02,  6.8413e-02,
   

### Forward pass
Now we will see in detail what happens when an input is plugged in the network, following each step we defined for our forward step

In [40]:
# First, create an input object and look into the model output 
# We don't have to call model.forward() directly
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
print(logits)
# To get a probability from this n-dim tensor, we can use softmax, which rescales it so that the elements of the n-dimensional output 
# tensor lie in the range [0,1] and sum to 1.
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

tensor([[-0.1312, -0.1306,  0.2446, -0.0676,  0.0721,  0.1156,  0.1606,  0.1262,
         -0.1219,  0.0984]], grad_fn=<AddmmBackward0>)
Predicted class: tensor([2])


Now let's have a closer look into what these layers and functions are doing

In [46]:
# Model Layers
input_image = torch.rand(1, 28, 28)
print(f"Initial image size : {input_image.size()}")

flatten = nn.Flatten()
flat_image = flatten(input_image)
print(f"Flattened image size : {flat_image.size()}")


Initial image size : torch.Size([1, 28, 28])
Flattened image size : torch.Size([1, 784])


In [47]:
# First network layer
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(f"Size of the output of the first hidden layer: {hidden1.size()}")

Size of the output of the first hidden layer: torch.Size([1, 20])


In [48]:
# Activation function
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.1171, -0.3113,  0.4478, -0.5449, -0.4348,  0.0035, -0.2499, -0.3078,
          0.3254,  0.2950,  0.1633,  0.0182,  0.1828, -0.2315,  0.0492, -0.2816,
         -0.0860, -0.3815,  0.5356, -0.6481]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.1171, 0.0000, 0.4478, 0.0000, 0.0000, 0.0035, 0.0000, 0.0000, 0.3254,
         0.2950, 0.1633, 0.0182, 0.1828, 0.0000, 0.0492, 0.0000, 0.0000, 0.0000,
         0.5356, 0.0000]], grad_fn=<ReluBackward0>)


In [49]:
# Define the complete model
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)

In [51]:
input_image = torch.rand(1, 28, 28)
logits = seq_modules(input_image)

print(f"Logits: {logits}")
# Softmax to get a probability
softmax_fn = nn.Softmax(dim=1)
pred_probab = softmax_fn(logits)
print(f"Predicted probabilities: {pred_probab}")
# Predict a class from the probabilities
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Logits: tensor([[ 0.1445,  0.1618,  0.3220,  0.0006, -0.2246,  0.1431, -0.2357, -0.0481,
         -0.1094,  0.0680]], grad_fn=<AddmmBackward0>)
Predicted probabilities: tensor([[0.1114, 0.1133, 0.1330, 0.0965, 0.0770, 0.1112, 0.0761, 0.0919, 0.0864,
         0.1032]], grad_fn=<SoftmaxBackward0>)
Predicted class: tensor([2])
